# Product Experimentation Case Study — Evaluating a Checkout Intervention

The decision question is whether a simplified checkout creates enough incremental customer and business value to ship. The analysis follows the pre-specified design: validate assignment, estimate the primary effect, examine guardrails, improve precision, then assess limited subgroup and time-dynamics evidence.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.statistics import proportion_effect, mean_effect, srm_test, required_sample_size, cuped_adjust
df = pd.read_csv(ROOT/'data/raw/experiment_users.csv', parse_dates=['assignment_timestamp'])
df['treatment'] = (df.experiment_group=='treatment').astype(int)
df.shape

(48000, 20)

## 1. Design and decision rules

- **Unit:** user, assigned once at the first eligible checkout start.
- **Primary metric:** purchase within 24 hours per assigned user.
- **Analysis:** intention to treat.
- **Duration:** fixed 28 days; no significance-based stopping.
- **Planning:** 45% baseline, 1.5 percentage-point MDE, 5% two-sided alpha, 80% power.
- **Business hurdle:** 0.75 percentage point. This is distinct from the statistical MDE.

In [2]:
required_sample_size(.45, .015, alpha=.05, power=.80)

{'per_arm': 17315, 'total': 34630}

## 2. Experiment health before outcomes

SRM, assignment balance, eligibility, missingness, and duration are checked before interpreting treatment effects. A passed SRM test does not prove flawless randomization; it removes one important warning sign.

In [3]:
counts = df.experiment_group.value_counts()
srm_test(counts['control'], counts['treatment'])

{'chi_square': 0.05633333333333333, 'p_value': 0.8123880360341355}

In [4]:
balance = pd.read_csv(ROOT/'data/processed/assignment_balance.csv')
balance.style.format({'control':'{:.3f}','treatment':'{:.3f}','standardized_difference':'{:.3f}'})

,variable,control,treatment,standardized_difference
0,Mobile share,0.671,0.682,0.024
1,Returning-user share,0.413,0.411,-0.004
2,Pre-period purchase,0.259,0.261,0.006
3,Pre-period sessions,2.036,2.065,0.018
4,Pre-period revenue,24.833,25.249,0.008


There is no evidence of SRM (p=0.812), required fields are complete, and the largest absolute standardized difference is 0.024. The realized sample exceeds the planned requirement.

## 3. Primary result

The primary estimate is the difference in user-level purchase proportions. The confidence interval is considered against both zero and the separate business hurdle.

In [5]:
c, t = df[df.treatment==0], df[df.treatment==1]
primary = proportion_effect(c.purchase_24h.sum(), len(c), t.purchase_24h.sum(), len(t))
pd.Series(primary)

control            0.470218
treatment          0.482311
absolute_effect    0.012093
relative_effect    0.025718
se                 0.004559
ci_low             0.003158
ci_high            0.021028
z                  2.652649
p_value            0.007986
dtype: float64

Treatment increased checkout completion from 47.02% to 48.23%: +1.21 percentage points (95% CI +0.32 to +2.10; p=0.008). The interval excludes zero but not the 0.75-point commercial hurdle. The experiment supports a positive effect, while the size that matters commercially remains uncertain.

## 4. Guardrails and business value

In [6]:
guardrails = pd.read_csv(ROOT/'data/processed/guardrail_effects.csv')
guardrails[['metric','control','treatment','absolute_effect','relative_effect','ci_low','ci_high','p_value']].style.format(precision=3)

,metric,control,treatment,absolute_effect,relative_effect,ci_low,ci_high,p_value
0,Net revenue / assigned user,44.579,44.555,-0.023,-0.001,-1.057,1.010,0.965
1,Average order value (buyers),98.818,96.924,-1.894,-0.019,-3.084,-0.705,0.002
2,Refunds / assigned user,0.019,0.022,0.003,0.184,0.001,0.006,0.007
3,Support contacts / assigned user,0.079,0.068,-0.011,-0.141,-0.016,-0.006,0.000
4,Checkout latency (ms),2181.214,1761.578,-419.636,-0.192,-426.971,-412.302,0.000


Latency and support contacts improve, but average order value falls 1.9% and refunds per assigned user rise 0.35 percentage points. Net revenue per assigned user is effectively unchanged, with a wide interval. These trade-offs prevent a conversion-only ship decision. AOV among buyers is conditional on a post-treatment event, so revenue per assigned user receives more decision weight.

## 5. CUPED variance reduction

The declared pre-period purchase indicator is correlated with the outcome but cannot be affected by treatment.

In [7]:
adjusted, theta = cuped_adjust(df.purchase_24h, df.pre_purchase_28d)
df['purchase_cuped'] = adjusted
unadjusted = proportion_effect(c.purchase_24h.sum(), len(c), t.purchase_24h.sum(), len(t))
adjusted_effect = mean_effect(df.loc[df.treatment==0,'purchase_cuped'], df.loc[df.treatment==1,'purchase_cuped'])
{'correlation':df.purchase_24h.corr(df.pre_purchase_28d), 'theta':theta, 'unadjusted_se':unadjusted['se'], 'adjusted_se':adjusted_effect['se'], 'adjusted_effect':adjusted_effect['absolute_effect']}

{'correlation': 0.10738219983665555,
 'theta': 0.12227996162721373,
 'unadjusted_se': 0.004558874465842159,
 'adjusted_se': 0.004532615610922644,
 'adjusted_effect': 0.011795116119165727}

CUPED reduces variance by only 1.15% because the pre-period covariate has weak correlation (0.107). It modestly improves precision and leaves the conclusion unchanged. CUPED cannot repair broken assignment, missing outcomes, or post-treatment bias.

## 6. Pre-specified heterogeneity

In [8]:
segments = pd.read_csv(ROOT/'data/processed/segment_effects.csv')
segments[['segment','value','control','treatment','absolute_effect','ci_low','ci_high','p_value']].style.format({'control':'{:.1%}','treatment':'{:.1%}','absolute_effect':'{:.2%}','ci_low':'{:.2%}','ci_high':'{:.2%}','p_value':'{:.3f}'})

,segment,value,control,treatment,absolute_effect,ci_low,ci_high,p_value
0,device_type,desktop,49.7%,50.1%,0.33%,-1.24%,1.90%,0.680
1,device_type,mobile,45.7%,47.4%,1.69%,0.60%,2.77%,0.002
2,new_vs_returning,new,42.2%,44.0%,1.74%,0.58%,2.89%,0.003
3,new_vs_returning,returning,53.8%,54.4%,0.51%,-0.88%,1.90%,0.469


Point estimates are larger for mobile and new users, but the direct interaction tests are not conclusive (p=0.163 and p=0.179). A significant estimate in one subgroup and a non-significant estimate in another does not establish a difference between them. The evidence does not support segment-specific shipping.

## 7. Time dynamics and decision

In [9]:
weekly = pd.read_csv(ROOT/'data/processed/weekly_effects.csv')
weekly[['week','control','treatment','absolute_effect','ci_low','ci_high']].style.format({'control':'{:.1%}','treatment':'{:.1%}','absolute_effect':'{:.2%}','ci_low':'{:.2%}','ci_high':'{:.2%}'})

,week,control,treatment,absolute_effect,ci_low,ci_high
0,1,46.3%,47.8%,1.44%,-0.36%,3.24%
1,2,47.9%,48.6%,0.66%,-1.13%,2.44%
2,3,46.4%,48.8%,2.31%,0.53%,4.08%
3,4,47.4%,47.8%,0.41%,-1.38%,2.20%


Weekly effects fluctuate, as expected, but there is no evidence of a linear treatment-by-day trend (p=0.894). The fixed-duration estimate is primary; repeatedly testing cumulative results and stopping at the first p<0.05 would inflate false positives.

**Decision: iterate and retest.** Preserve the latency and support improvements, investigate the refund increase and order-value decline, and test a revised checkout. Do not ship broadly or claim a mobile-only win from these subgroup results.